In [1]:
!pip install pysrt pydub tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pysrt: filename=pysrt-1.1.2-py3-none-any.whl size=13443 sha256=4489148d11a93fd1ae8658b63ea8d782a4f3bfc9bd76d5cae9ce9ba8f4a95b0e
  Stored in directory: /root/.cache/pip/wheels/6a/36/54/2aa8dc961885dfa7b0ebd45a57505f25039d79b4ea0fd9f29d
Successfully built pysrt


In [2]:
from google.colab import files

uploaded = files.upload()

Saving A1_A_good_nights_sleep.mp3 to A1_A_good_nights_sleep (1).mp3
Saving A1_A_good_nights_sleep.srt to A1_A_good_nights_sleep (1).srt


In [3]:
MIN_WORDS = 5
MAX_WORDS = 12

MIN_DURATION = 2.0
MAX_DURATION = 6.0

LONG_PAUSE = 0.7

MIN_CHARS = 20

In [4]:
import re

BAD_ENDINGS = {
    "a","an","the",
    "of","in","on","at","for","with","from",
    "to","by","about","into","through",
    "is","are","was","were",
    "have","has","had",
    "do","does","did",
    "can","could","will","would",
    "should","may","might",
    "and","or","but"
}


def word_count(text):
    return len(text.split())


def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def good_ending(text):
    """
    Determina si es un buen lugar para terminar.
    """

    text = text.strip()

    if not text:
        return False

    last_word = text.split()[-1].lower()
    last_word = re.sub(r"[^\w]", "", last_word)

    if last_word in BAD_ENDINGS:
        return False

    if text.endswith((".", "!", "?")):
        return True

    if text.endswith(","):
        return True

    return False

In [5]:
import pysrt

SRT_FILE = next(
    f for f in uploaded.keys()
    if f.lower().endswith(".srt")
)

subs = pysrt.open(SRT_FILE)

print("Subtítulos cargados:", len(subs))

Subtítulos cargados: 34


In [6]:
segments = []

current_text = []
current_start = None
current_end = None

for i, sub in enumerate(subs):

    text = clean_text(sub.text.replace("\n", " "))

    start = sub.start.ordinal / 1000
    end = sub.end.ordinal / 1000

    if current_start is None:
        current_start = start

    current_end = end
    current_text.append(text)

    merged = " ".join(current_text)

    duration = current_end - current_start
    words = word_count(merged)

    next_pause = 0

    if i < len(subs)-1:
        next_start = subs[i+1].start.ordinal / 1000
        next_pause = next_start - end

    can_close = (
        duration >= MIN_DURATION
        and words >= MIN_WORDS
        and len(merged) >= MIN_CHARS
    )

    preferred_break = (
        good_ending(merged)
        or next_pause >= LONG_PAUSE
    )

    force_break = (
        duration >= MAX_DURATION
        or words >= MAX_WORDS
    )

    if can_close and (preferred_break or force_break):

        segments.append({
            "start": current_start,
            "end": current_end,
            "text": merged
        })

        current_text = []
        current_start = None
        current_end = None

# resto pendiente

if current_text:

    merged = " ".join(current_text)

    segments.append({
        "start": current_start,
        "end": current_end,
        "text": merged
    })

print("Segmentos creados:", len(segments))

Segmentos creados: 22


In [7]:
for i, seg in enumerate(segments[:20], 1):

    duration = seg["end"] - seg["start"]

    print("="*50)
    print(i)
    print("Duración:", round(duration,2))
    print(seg["text"])

1
Duración: 7.94
At exam time, it is important to sleep well. Today we have Dr. Baker with
2
Duración: 3.58
us in the studio, and he is going to give us 5
3
Duración: 4.0
top tips for getting a good night's sleep.
4
Duración: 5.56
Welcome to the show, Dr. Baker. Thank you. It's great to be here.
5
Duración: 3.66
Let's start with tip 1.
6
Duración: 2.98
Don't go to bed with the television on.
7
Duración: 7.64
Some people think they can sleep well with the TV on, but the noise and the lights mean you don't really sleep well.
8
Duración: 3.52
So turn it off. Tip 2.
9
Duración: 4.8
Tip 2: Don't think too much before bedtime.
10
Duración: 5.67
Do your hardest homework earlier in the evening. Do easier homework later. If
11
Duración: 2.9
your brain is too busy and full of ideas,
12
Duración: 2.01
it takes longer to get to sleep.
13
Duración: 6.74
Tip 3: Don't play video games for an hour before you go to sleep.
14
Duración: 4.07
They also make your brain too busy and active.
15
Duración: 5.9

In [8]:
import os
import shutil

OUTPUT_DIR = "output_files"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR)

In [9]:
from pydub import AudioSegment
from tqdm import tqdm

MP3_FILE = next(
    f for f in uploaded.keys()
    if f.lower().endswith(".mp3")
)

audio = AudioSegment.from_mp3(MP3_FILE)

for idx, seg in enumerate(tqdm(segments), start=1):

    start_ms = int(seg["start"] * 1000)
    end_ms = int(seg["end"] * 1000)

    clip = audio[start_ms:end_ms]

    filename = f"audio_{idx:04d}.mp3"

    clip.export(
        os.path.join(OUTPUT_DIR, filename),
        format="mp3"
    )

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
100%|██████████| 22/22 [00:06<00:00,  3.40it/s]


In [10]:
import csv

tsv_path = os.path.join(
    OUTPUT_DIR,
    "deck.tsv"
)

with open(
    tsv_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f,
        delimiter="\t"
    )

    for idx, seg in enumerate(segments, start=1):

        audio_name = f"audio_{idx:04d}.mp3"

        writer.writerow([
            seg["text"],
            "",
            f"[sound:{audio_name}]"
        ])

print("TSV creado")

TSV creado


In [11]:
import shutil

zip_name = "output_files"

shutil.make_archive(
    zip_name,
    "zip",
    OUTPUT_DIR
)

print("ZIP creado")

ZIP creado


In [12]:
from google.colab import files

files.download(
    "output_files.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>